# Multi-Cycle Experiment Pipeline: Hist-RTSGA



In [ ]:
# ======================================================
# Cell 1: Environment Setup — Run this first!
# ======================================================
# Detects Google Colab vs local Jupyter and sets three variables used
# throughout the notebook:
#
#   DATA_FILE     — absolute path to Data/mapped-dataset-36-20.xlsx
#   OUTPUT_DIR    — directory where results/checkpoints are saved
#   BASELINE_FILE — path to summary_statistics_RTSGA_only_RTW_10%.xlsx
#                   (produced by the RTSGA notebook; used for comparison plots)
#
# ── Google Colab users ──────────────────────────────────────────────
#   Upload the project to Google Drive, then set DRIVE_PROJECT_PATH
#   below to the folder that contains the Data/ subfolder.
#
# ── Local Jupyter users ─────────────────────────────────────────────
#   No action needed. Paths are found automatically by walking up from
#   the current working directory until Data/mapped-dataset-36-20.xlsx
#   is found.
# ======================================================

import subprocess, sys, os
from pathlib import Path

# ── Install dependencies ──────────────────────────────────────────────
for _pkg in ["pandas", "numpy", "matplotlib", "seaborn", "scipy", "openpyxl", "ipywidgets"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", _pkg, "-q"])
print("✅ All dependencies installed.")

# ── Detect environment ────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Resolve paths ─────────────────────────────────────────────────────
_DATA_FILENAME = "mapped-dataset-36-20.xlsx"

if IN_COLAB:
    # ─── COLAB — Option A: Google Drive (recommended) ─────────────────
    # Set DRIVE_PROJECT_PATH to the Drive folder that contains Data/, RTSGA/, and Hist-RTSGA/
    DRIVE_PROJECT_PATH = "/content/drive/MyDrive/BV-Driven-History-RTSGA"  # ← adjust if needed

    try:
        from google.colab import drive
        drive.mount("/content/drive")
        _data_candidate = Path(DRIVE_PROJECT_PATH) / "Data" / _DATA_FILENAME
        if not _data_candidate.exists():
            raise FileNotFoundError(
                f"{_DATA_FILENAME} not found at {_data_candidate}. "
                "Check DRIVE_PROJECT_PATH and make sure you uploaded the project to Drive."
            )
        DATA_FILE     = str(_data_candidate)
        OUTPUT_DIR    = str(Path(DRIVE_PROJECT_PATH) / "Hist-RTSGA")
        BASELINE_FILE = str(Path(DRIVE_PROJECT_PATH) / "RTSGA" / "summary_statistics_RTSGA_only_RTW_10%.xlsx")

    except Exception as _e:
        # ─── COLAB — Option B: File upload fallback ────────────────────
        print(f"Drive mount failed ({_e}).\nFalling back to manual file upload.")
        from google.colab import files as _cf

        print(f"\nStep 1 — Upload {_DATA_FILENAME}:")
        _up = _cf.upload()
        DATA_FILE = next(iter(_up))

        OUTPUT_DIR = "/content"

        print("\nStep 2 — Upload summary_statistics_RTSGA_only_RTW_10%.xlsx (Cancel to skip):")
        try:
            _up2 = _cf.upload()
            BASELINE_FILE = next(iter(_up2)) if _up2 else None
        except Exception:
            BASELINE_FILE = None

else:
    # ─── LOCAL — walk up from CWD to find the project root ────────────
    def _find_root(start: Path, marker: str):
        for p in [start] + list(start.parents):
            if (p / marker).exists():
                return p
        return None

    _root = _find_root(Path.cwd(), os.path.join("Data", _DATA_FILENAME))
    if _root is None:
        raise FileNotFoundError(
            f"Could not find Data/{_DATA_FILENAME} by walking up from the current directory.\n"
            "Open this notebook from inside the project folder, or set DATA_FILE manually."
        )
    DATA_FILE  = str(_root / "Data" / _DATA_FILENAME)
    OUTPUT_DIR = str(Path.cwd())   # outputs (checkpoints, plots, etc.) saved next to the notebook

    # The RTSGA baseline lives in the RTSGA/ sibling folder; fall back to OUTPUT_DIR if absent
    _rtsga_baseline = _root / "RTSGA" / "summary_statistics_RTSGA_only_RTW_10%.xlsx"
    BASELINE_FILE   = str(_rtsga_baseline) if _rtsga_baseline.exists() \
                      else str(Path.cwd() / "summary_statistics_RTSGA_only_RTW_10%.xlsx")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\nEnvironment  : {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"Data file    : {DATA_FILE}")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"Baseline file: {BASELINE_FILE}")

In [12]:
# !pip install openpyxl
# ======================================================
# Cell 1: Imports and Setup
# ======================================================

import pandas as pd
import numpy as np
import random
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Experiment Configuration
RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

NUM_CYCLES = 20
RUNS_PER_CYCLE = 30
RTW_RATIOS = [0.1]  # 5% and 10%

# GA Parameters
GA_PARAMS = {
    'max_generations': 100,
    'crossover_prob': 0.8,
    'mutation_prob': 0.05,
    'population_size': 10
}

print("Setup complete. Ready for multi-cycle experiments.")
print(f"Configuration: {NUM_CYCLES} cycles, {RUNS_PER_CYCLE} runs per cycle")
print(f"RTW ratios: {[f'{r*100}%' for r in RTW_RATIOS]}")

Setup complete. Ready for multi-cycle experiments.
Configuration: 20 cycles, 30 runs per cycle
RTW ratios: ['10.0%']


In [ ]:
# ======================================================
# Cell 2: Data Loading and Preparation
# ======================================================

# Load dataset using DATA_FILE set by Cell 1
filename = DATA_FILE
df = pd.read_excel(filename)

# Normalize column names
df.columns = [col.strip().lower() for col in df.columns]

# Create data maps
data_maps = {
    'req_to_tests': df.groupby('us_id')['tc_id'].apply(set).to_dict(),
    'req_to_bv': df.groupby('us_id')['us_businessvalue'].first().to_dict(),
    'test_to_time': df.groupby('tc_id')['tc_executiontime'].first().to_dict(),
    'all_req_ids': sorted(df['us_id'].unique()),
    'tests': sorted(df['tc_id'].unique())
}

data_maps['n_reqs'] = len(data_maps['all_req_ids'])
data_maps['n_tests'] = len(data_maps['tests'])

# Calculate total execution time
TOTAL_EXEC_TIME = df.drop_duplicates(subset=['tc_id'])['tc_executiontime'].sum()

print(f"Dataset loaded: {data_maps['n_tests']} tests, {data_maps['n_reqs']} requirements")
print(f"Total execution time: {TOTAL_EXEC_TIME:.2f}")
print(f"Requirements range: {min(data_maps['all_req_ids'])} to {max(data_maps['all_req_ids'])}")
print(f"Business values range: {min(data_maps['req_to_bv'].values())} to {max(data_maps['req_to_bv'].values())}")

In [14]:
# ======================================================
# Cell 3: Algorithm Implementations (Updated with Starvation Logic)
# ======================================================
import random
import numpy as np

# ======================================================
# The GA Logic as a Reusable Function (Updated)
# ======================================================
def run_ga_instance(max_exec_time, ga_params, data_maps):
    # Unpack GA parameters and data maps
    max_generations = ga_params['max_generations']
    crossover_prob = ga_params['crossover_prob']
    mutation_prob = ga_params['mutation_prob']

    req_to_tests = data_maps['req_to_tests']
    req_to_bv = data_maps['req_to_bv']
    test_to_time = data_maps['test_to_time']
    all_req_ids = data_maps['all_req_ids']
    # bval_to_reqs_map = data_maps['bval_to_reqs_map']  # kept for compatibility if needed
    n_requirements = len(all_req_ids)

    # ----------------------------------------------------------------------
    # New: starvation-related configuration (all optional)
    # ----------------------------------------------------------------------
    # req_to_starvation: requirement -> starvation counter (cycles since last covered)
    req_to_starvation = data_maps.get('req_to_starvation', {})
    # bv_tolerance: fraction of best BV we are willing to accept (e.g., 0.9 = 90%)
    bv_tolerance = ga_params.get('bv_tolerance', 1.0)
    # starvation_weight: strength of starvation bonus when BV is close to best
    starvation_weight = ga_params.get('starvation_weight', 0.0)

    # Track best BV seen (estimated baseline) to apply tolerance inside fitness
    baseline_tracker = {'best_bv_seen': 0.0}

    # ----------------------------------------------------------------------
    # Helper: Evaluation Decomposition (Deb, 2000)
    # Source: Deb (2000), "An Efficient Constraint Handling Method for GAs"
    # ----------------------------------------------------------------------
    def evaluate_chrom(chrom):
        covered_tests = set()
        total_bv = 0
        for idx, val in enumerate(chrom):
            if val:
                req = all_req_ids[idx]
                covered_tests |= req_to_tests.get(req, set())
                total_bv += req_to_bv.get(req, 0)
        total_time = sum(test_to_time.get(t, 0) for t in covered_tests)
        feasible = (total_time <= max_exec_time)
        violation = max(0, total_time - max_exec_time)
        return total_bv, total_time, feasible, violation

    # ----------------------------------------------------------------------
    # New: starvation relief score
    # ----------------------------------------------------------------------
    def starvation_relief(chrom):
        """
        Simple starvation relief: sum of starvation counters
        for all selected requirements. Higher = more relief.
        """
        if not req_to_starvation:
            return 0.0
        relief_score = 0.0
        for idx, val in enumerate(chrom):
            if val:
                req = all_req_ids[idx]
                relief_score += float(req_to_starvation.get(req, 0.0))
        return relief_score

    # ----------------------------------------------------------------------
    # Feasible Initialization (Chu & Beasley, 1998)
    # ----------------------------------------------------------------------
    def feasible_chromosome():
        chrom = [0] * n_requirements
        indices = list(range(n_requirements))
        random.shuffle(indices)
        covered_tests = set()
        total_time = 0
        for idx in indices:
            req = all_req_ids[idx]
            tests = req_to_tests.get(req, set())
            new_tests = tests - covered_tests
            additional_time = sum(test_to_time.get(t, 0) for t in new_tests)
            if total_time + additional_time <= max_exec_time:
                chrom[idx] = 1
                covered_tests |= new_tests
                total_time += additional_time
        return chrom

    population_size = ga_params.get('population_size', 10)
    population = [feasible_chromosome() for _ in range(population_size)]

    # ----------------------------------------------------------------------
    # Greedy Seeding for Budgeted BV Coverage (Khuller, Moss, Naor, 1999)
    # Source: Khuller–Moss–Naor (1999), Budgeted Maximum Coverage (1−1/e)
    # ----------------------------------------------------------------------
    def greedy_seed_variant():
        chrom = [0] * n_requirements
        covered = set()
        used_time = 0
        remaining = set(range(n_requirements))
        while remaining:
            best_idx, best_score = None, float("-inf")
            for idx in remaining:
                req = all_req_ids[idx]
                tests = req_to_tests.get(req, set())
                new_tests = tests - covered
                add_time = sum(test_to_time.get(t, 0) for t in new_tests)
                if add_time < 0:
                    continue
                score = req_to_bv.get(req, 0) if add_time == 0 else (
                    req_to_bv.get(req, 0) / (add_time + 1e-12)
                )
                if used_time + add_time <= max_exec_time and score > best_score:
                    best_idx, best_score = idx, score
            if best_idx is None:
                break
            req = all_req_ids[best_idx]
            new_tests = req_to_tests.get(req, set()) - covered
            add_time = sum(test_to_time.get(t, 0) for t in new_tests)
            if used_time + add_time <= max_exec_time:
                chrom[best_idx] = 1
                covered |= new_tests
                used_time += add_time
            remaining.remove(best_idx)
        return chrom

    # Inject a few seeds to improve the starting population
    num_seeds = min(3, population_size)
    for i in range(num_seeds):
        population[i] = greedy_seed_variant()

    # ----------------------------------------------------------------------
    # Value-Aware Repair / Improve Operator
    # Source: Chu & Beasley (1998); randomized/efficiency-based repair literature
    # ----------------------------------------------------------------------
    def repair_value_aware(chrom):
        chrom = chrom[:]  # work on a copy

        # DROP phase: remove the least efficient requirement until feasible
        while True:
            _, _, feasible, _ = evaluate_chrom(chrom)
            if feasible:
                break
            selected = [i for i, v in enumerate(chrom) if v]
            if not selected:
                break
            worst_idx, worst_ratio = None, float("inf")
            for idx in selected:
                req = all_req_ids[idx]
                covered_other = set()
                for j, v in enumerate(chrom):
                    if v and j != idx:
                        covered_other |= req_to_tests.get(all_req_ids[j], set())
                unique_tests = req_to_tests.get(req, set()) - covered_other
                time_save = sum(test_to_time.get(t, 0) for t in unique_tests)
                bv_loss = req_to_bv.get(req, 0)
                ratio = bv_loss / (time_save + 1e-12) if time_save > 0 else float("inf")
                ratio += random.uniform(-1e-9, 1e-9)  # tie-breaking noise
                if ratio < worst_ratio:
                    worst_ratio, worst_idx = ratio, idx
            if worst_idx is None:
                break
            chrom[worst_idx] = 0

        # ADD phase: greedily add the most efficient requirement that fits
        while True:
            _, tt, _, _ = evaluate_chrom(chrom)
            slack = max_exec_time - tt
            if slack <= 0:
                break
            candidates = [i for i, v in enumerate(chrom) if not v]
            best_idx, best_score = None, float("-inf")
            covered_now = set()
            for j, v in enumerate(chrom):
                if v:
                    covered_now |= req_to_tests.get(all_req_ids[j], set())
            for idx in candidates:
                req = all_req_ids[idx]
                new_tests = req_to_tests.get(req, set()) - covered_now
                add_time = sum(test_to_time.get(t, 0) for t in new_tests)
                if add_time < 0:
                    continue
                score = req_to_bv.get(req, 0) if add_time == 0 else (
                    req_to_bv.get(req, 0) / (add_time + 1e-12)
                )
                score += random.uniform(-1e-9, 1e-9)
                if add_time <= slack and score > best_score:
                    best_idx, best_score = idx, score
            if best_idx is None:
                break
            chrom[best_idx] = 1

        return chrom

    # ----------------------------------------------------------------------
    # Fitness Function (Yoo & Harman, 2012, Sec. 5.1) + starvation extension
    # ----------------------------------------------------------------------
    def fitness(chrom):
        total_bv, _, _, _ = evaluate_chrom(chrom)

        # If no starvation handling configured, behave like original GA.
        if starvation_weight <= 0.0:
            return total_bv

        best_bv_seen = baseline_tracker['best_bv_seen']

        # Only reward starvation when this chromosome's BV is close to best BV.
        if best_bv_seen > 0.0 and bv_tolerance < 1.0:
            min_acceptable_bv = bv_tolerance * best_bv_seen
            if total_bv < min_acceptable_bv:
                # Too far from best BV: ignore starvation bonus.
                return total_bv

        relief_score = starvation_relief(chrom)
        # Combined score: BV is primary, starvation_relief is secondary.
        return total_bv + starvation_weight * relief_score

    # ----------------------------------------------------------------------
    # Deb-Style Tournament Selection (Deb, 2000)
    # ----------------------------------------------------------------------
    def deb_better(a_chrom, b_chrom):
        # Evaluate both chromosomes
        a_bv, _, a_feas, a_viol = evaluate_chrom(a_chrom)
        b_bv, _, b_feas, b_viol = evaluate_chrom(b_chrom)

        # Feasibility first
        if a_feas and not b_feas:
            return True
        if b_feas and not a_feas:
            return False

        if a_feas and b_feas:
            # Both feasible: compare by combined fitness (BV + starvation)
            return fitness(a_chrom) > fitness(b_chrom)

        # Both infeasible: lower violation is better
        return a_viol < b_viol

    def deb_tournament_selection(population, k=3):
        contestants = random.sample(population, k=min(k, len(population)))
        best = contestants[0]
        for c in contestants[1:]:
            if deb_better(c, best):
                best = c
        return best

    # ----------------------------------------------------------------------
    # Crossover and Mutation Operators (Mitchell, 1998) - Unchanged
    # ----------------------------------------------------------------------
    def single_point_crossover(parent1, parent2):
        if n_requirements < 2:
            return parent1[:], parent2[:]
        point = random.randint(1, n_requirements - 1)
        child1 = parent1[:point] + parent2[point:]
        child2 = parent2[:point] + parent1[point:]
        return child1, child2

    def mutate(chromosome, mutation_prob=mutation_prob):
        if random.random() < mutation_prob and n_requirements > 0:
            mutation_idx = random.randint(0, n_requirements - 1)
            chromosome[mutation_idx] = 1 - chromosome[mutation_idx]

    # ----------------------------------------------------------------------
    # Initialize baseline best BV from the initial population
    # ----------------------------------------------------------------------
    if population:
        initial_bvs = [evaluate_chrom(ch)[0] for ch in population]
        if initial_bvs:
            baseline_tracker['best_bv_seen'] = max(initial_bvs)

    # ----------------------------------------------------------------------
    # Main GA Loop (modified to use new operators + baseline tracking)
    # ----------------------------------------------------------------------
    for generation in range(max_generations):
        # --- Selection ---
        mating_pool = []
        for _ in range(population_size // 2):
            parent1 = deb_tournament_selection(population)
            parent2 = deb_tournament_selection(population)
            mating_pool.append((parent1, parent2))

        # --- Variation & Repair ---
        offspring = []
        for parent1, parent2 in mating_pool:
            if random.random() < crossover_prob:
                child1, child2 = single_point_crossover(parent1, parent2)
            else:
                child1, child2 = parent1[:], parent2[:]
            mutate(child1)
            mutate(child2)
            child1 = repair_value_aware(child1)
            child2 = repair_value_aware(child2)
            offspring.extend([child1, child2])

        # --- Replacement (using Deb's rules + combined fitness for ranking) ---
        combined_population = population + offspring

        def deb_key(ch):
            total_bv, _, feas, viol = evaluate_chrom(ch)
            if feas:
                # Feasible: prioritize lower violation, then higher combined fitness
                return (0, viol, -fitness(ch))
            # Infeasible: prioritize lower violation, then higher BV
            return (1, viol, -total_bv)

        combined_population.sort(key=deb_key)
        population = combined_population[:population_size]

        # Update best BV seen so far (for BV tolerance)
        best_bv_gen, _, _, _ = evaluate_chrom(population[0])
        if best_bv_gen > baseline_tracker['best_bv_seen']:
            baseline_tracker['best_bv_seen'] = best_bv_gen

    # Final best solution is the top of the population after the last sort
    best_solution = population[0]
    best_fitness, _, _, _ = evaluate_chrom(best_solution)  # this is pure BV
    selected_reqs = [all_req_ids[i] for i, val in enumerate(best_solution) if val]

    return selected_reqs, best_fitness

# ----- HISTORY-AWARE RTSGA (Updated to calculate Starvation) -----
def run_hist_rtsga_cycles(rtw_ratio, data_maps, ga_params, num_cycles=20, runs_per_cycle=30):
    """Run history-aware RTSGA with starvation tracking and sqrt penalty"""
    max_exec_time = rtw_ratio * TOTAL_EXEC_TIME
    req_to_bv_orig = data_maps['req_to_bv']
    all_req_ids = data_maps['all_req_ids']

    # Initialize selection counts (for SQRT penalty)
    select_counts = {req: 0 for req in req_to_bv_orig}

    # Initialize starvation counts (Cycles since last selected)
    # Initially 0 (or you could start higher to encourage early exploration)
    starvation_counts = {req: 0 for req in all_req_ids}

    cycles_results = []

    for cycle_idx in range(1, num_cycles + 1):
        # 1. Apply SQRT PENALTY to BV based on selection history
        req_to_bv_for_cycle = {
            req: req_to_bv_orig[req] / np.sqrt(1 + select_counts[req])
            for req in req_to_bv_orig
        }
        # print(req_to_bv_for_cycle)

        # 2. Prepare Data Maps (include starvation counts)
        data_maps_for_cycle = data_maps.copy()
        data_maps_for_cycle['req_to_bv'] = req_to_bv_for_cycle
        data_maps_for_cycle['req_to_starvation'] = starvation_counts.copy()

        # Run multiple independent runs for this cycle
        all_runs = []
        for run_idx in range(runs_per_cycle):
            selected_reqs, best_fitness = run_ga_instance(
                max_exec_time=max_exec_time,
                ga_params=ga_params,
                data_maps=data_maps_for_cycle
            )
            all_runs.append({
                'selected_reqs': selected_reqs,
                'fitness': best_fitness
            })

        # Find best run
        best_run = max(all_runs, key=lambda r: r['fitness'])

        cycles_results.append({
            'cycle_idx': cycle_idx,
            'best_selected_reqs': best_run['selected_reqs'],
            'best_bv': best_run['fitness'],
            'all_runs': all_runs
        })

        # Update History for next cycle
        selected_set = set(best_run['selected_reqs'])

        for req in all_req_ids:
            if req in selected_set:
                # Selected: Reset starvation, increment selection count
                select_counts[req] += 1
                starvation_counts[req] = 0
            else:
                # Not selected: Increment starvation count
                starvation_counts[req] += 1

    return cycles_results

# ----- STANDARD RTSGA (No History) -----
def run_rtsga_cycles(rtw_ratio, data_maps, ga_params, num_cycles=20, runs_per_cycle=30):
    """Run standard RTSGA without history awareness"""
    max_exec_time = rtw_ratio * TOTAL_EXEC_TIME

    cycles_results = []

    for cycle_idx in range(1, num_cycles + 1):
        # No BV modification - use original values
        all_runs = []
        for run_idx in range(runs_per_cycle):
            selected_reqs, best_fitness = run_ga_instance(
                max_exec_time=max_exec_time,
                ga_params=ga_params,
                data_maps=data_maps
            )
            all_runs.append({
                'selected_reqs': selected_reqs,
                'fitness': best_fitness
            })

        best_run = max(all_runs, key=lambda r: r['fitness'])
        cycles_results.append({
            'cycle_idx': cycle_idx,
            'best_selected_reqs': best_run['selected_reqs'],
            'best_bv': best_run['fitness'],
            'all_runs': all_runs
        })

    return cycles_results

print("Algorithms updated: Hist-RTSGA now tracks starvation.")

Algorithms updated: Hist-RTSGA now tracks starvation.


In [ ]:
# ======================================================
# Cell 4: Multi-Cycle Experiments with Checkpointing
# ======================================================
import time
import os
import pickle

# Checkpoint directory inside OUTPUT_DIR (set by Cell 1)
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "Checkpoints")

if not os.path.exists(CHECKPOINT_DIR):
    os.makedirs(CHECKPOINT_DIR)
    print(f"Created checkpoint directory: {CHECKPOINT_DIR}")
else:
    print(f"Using existing checkpoint directory: {CHECKPOINT_DIR}")

results_storage = {}

# Experiment Configurations
rtw_target = 0.10
bv_tolerances = [0.90, 0.80, 0.70]
starvation_weights = [0.05, 0.10, 0.15, 0.20, 0.25]

print(f"Starting Starvation Sensitivity Experiments at RTW={rtw_target*100}%...")
print(f"Cycles: {NUM_CYCLES}, Runs per Cycle: {RUNS_PER_CYCLE}")

for tol in bv_tolerances:
    for weight in starvation_weights:
        # Create unique label
        config_label = f"Tol{tol}_Wt{weight}"
        checkpoint_file = os.path.join(CHECKPOINT_DIR, f"{config_label}.pkl")

        print(f"\n--- Checking Config: {config_label} ---")

        # --- CHECKPOINT LOGIC ---
        if os.path.exists(checkpoint_file):
            print(f"Found checkpoint for {config_label}. Loading...")
            try:
                with open(checkpoint_file, 'rb') as f:
                    cycle_results = pickle.load(f)
                results_storage[config_label] = cycle_results
                print("Loaded successfully. Skipping re-computation.")
                continue
            except Exception as e:
                print(f"Error loading checkpoint (file might be corrupted): {e}")
                print("Re-running this configuration...")

        # --- EXECUTION LOGIC (If no checkpoint) ---
        print(f"No checkpoint found. Running Hist-RTSGA...")

        current_ga_params = GA_PARAMS.copy()
        current_ga_params['bv_tolerance'] = tol
        current_ga_params['starvation_weight'] = weight

        start_time = time.time()

        cycle_results = run_hist_rtsga_cycles(
            rtw_ratio=rtw_target,
            data_maps=data_maps,
            ga_params=current_ga_params,
            num_cycles=NUM_CYCLES,
            runs_per_cycle=RUNS_PER_CYCLE
        )

        elapsed = time.time() - start_time
        print(f"Completed {config_label} in {elapsed:.2f}s")

        results_storage[config_label] = cycle_results

        try:
            with open(checkpoint_file, 'wb') as f:
                pickle.dump(cycle_results, f)
            print(f"Checkpoint saved to: {checkpoint_file}")
        except Exception as e:
            print(f"WARNING: Could not save checkpoint: {e}")

print("\n" + "="*70)
print("All combinations completed/loaded successfully!")
print("Keys available in results_storage:", list(results_storage.keys()))
print("="*70)

In [ ]:
# ======================================================
# Cell 5: Create Selection History Tables
# ======================================================
import pandas as pd
import os

# Results directory inside OUTPUT_DIR (set by Cell 1)
RESULTS_DIR = os.path.join(OUTPUT_DIR, "History_Tables")

if not os.path.exists(RESULTS_DIR):
    os.makedirs(RESULTS_DIR)
    print(f"Created results directory: {RESULTS_DIR}")
else:
    print(f"Using results directory: {RESULTS_DIR}")

def create_selection_history_table(cycles_data, config_name):
    """
    Create table showing (req_id, BV) pairs for the best run in each cycle.
    """
    rows = []
    row = {'Configuration': config_name}

    for cycle_result in cycles_data:
        cycle_idx = cycle_result['cycle_idx']
        best_reqs = cycle_result['best_selected_reqs']

        req_bv_pairs = [(req, data_maps['req_to_bv'][req]) for req in best_reqs]
        req_bv_str = ', '.join([f"({r},{bv})" for r, bv in req_bv_pairs])
        row[f'C{cycle_idx}'] = req_bv_str

    rows.append(row)
    return pd.DataFrame(rows)

print("\nGenerating Selection History Files...")
print("="*60)

all_configs_history_list = []

for config_label in results_storage.keys():
    history_table = create_selection_history_table(results_storage[config_label], config_label)
    all_configs_history_list.append(history_table)

    history_filename = f'selection_history_{config_label}.xlsx'
    full_path = os.path.join(RESULTS_DIR, history_filename)
    history_table.to_excel(full_path, index=False)
    print(f"Saved Individual: {full_path}")

if all_configs_history_list:
    combined_history_df = pd.concat(all_configs_history_list, ignore_index=True)
    combined_full_path = os.path.join(RESULTS_DIR, "selection_history_ALL_COMBINED.xlsx")
    combined_history_df.to_excel(combined_full_path, index=False)
    print("\n" + "="*60)
    print(f"Saved Consolidated File: {combined_full_path}")

print("All selection history tables created successfully!")

In [ ]:
# ======================================================
# Cell 6: Calculate Summary Statistics
# ======================================================
import numpy as np
import pandas as pd
import os

# Results directory inside OUTPUT_DIR (set by Cell 1)
RESULTS_DIR = os.path.join(OUTPUT_DIR, "Statistics")

if not os.path.exists(RESULTS_DIR):
    os.makedirs(RESULTS_DIR)
    print(f"Created results directory: {RESULTS_DIR}")
else:
    print(f"Using results directory: {RESULTS_DIR}")

if 'n_reqs' not in data_maps:
    data_maps['n_reqs'] = len(data_maps['all_req_ids'])
if 'n_tests' not in data_maps:
    data_maps['n_tests'] = len(data_maps['test_to_time'])

def calculate_cycle_statistics(cycles_data, data_maps):
    method_stats = []

    for cycle_result in cycles_data:
        cycle_idx = cycle_result['cycle_idx']
        all_runs = cycle_result['all_runs']

        bv_values = []
        req_cov_values = []
        test_cov_values = []

        for run in all_runs:
            selected_reqs = run['selected_reqs']

            total_bv = run['fitness']
            bv_values.append(total_bv)

            req_cov = (len(selected_reqs) / data_maps['n_reqs']) * 100
            req_cov_values.append(req_cov)

            covered_tests = set()
            for req in selected_reqs:
                covered_tests |= data_maps['req_to_tests'].get(req, set())
            test_cov = (len(covered_tests) / data_maps['n_tests']) * 100
            test_cov_values.append(test_cov)

        cycle_stats = {
            'cycle': cycle_idx,
            'bv_mean': np.mean(bv_values),
            'bv_std': np.std(bv_values),
            'bv_median': np.median(bv_values),
            'bv_max': np.max(bv_values),
            'bv_min': np.min(bv_values),
            'req_cov_mean': np.mean(req_cov_values),
            'req_cov_std': np.std(req_cov_values),
            'req_cov_median': np.median(req_cov_values),
            'req_cov_max': np.max(req_cov_values),
            'req_cov_min': np.min(req_cov_values),
            'test_cov_mean': np.mean(test_cov_values),
            'test_cov_std': np.std(test_cov_values),
            'test_cov_median': np.median(test_cov_values),
            'test_cov_max': np.max(test_cov_values),
            'test_cov_min': np.min(test_cov_values)
        }
        method_stats.append(cycle_stats)

    return pd.DataFrame(method_stats)


def create_summary_table(df_stats, config_name, metric='bv'):
    rows = []
    row = {'Configuration': config_name}

    for _, cycle_row in df_stats.iterrows():
        cycle_idx = int(cycle_row['cycle'])
        max_val = cycle_row[f'{metric}_max']
        min_val = cycle_row[f'{metric}_min']
        std_val = cycle_row[f'{metric}_std']
        row[f'C{cycle_idx}'] = f"({max_val:.2f}, {min_val:.2f}, {std_val:.2f})"

    rows.append(row)
    return pd.DataFrame(rows)


all_statistics = {}
consolidated_bv = []
consolidated_req = []
consolidated_test = []

print("Calculating Statistics...")
print("="*60)

for config_label in results_storage.keys():
    print(f"Processing: {config_label}")

    df_stats = calculate_cycle_statistics(results_storage[config_label], data_maps)
    all_statistics[config_label] = df_stats

    bv_table = create_summary_table(df_stats, config_label, 'bv')
    req_cov_table = create_summary_table(df_stats, config_label, 'req_cov')
    test_cov_table = create_summary_table(df_stats, config_label, 'test_cov')

    consolidated_bv.append(bv_table)
    consolidated_req.append(req_cov_table)
    consolidated_test.append(test_cov_table)

    filename = f'summary_statistics_{config_label}.xlsx'
    full_path = os.path.join(RESULTS_DIR, filename)

    with pd.ExcelWriter(full_path, engine='openpyxl') as writer:
        bv_table.to_excel(writer, sheet_name='BV_Summary', index=False)
        req_cov_table.to_excel(writer, sheet_name='Req_Cov_Summary', index=False)
        test_cov_table.to_excel(writer, sheet_name='Test_Cov_Summary', index=False)
        df_stats.to_excel(writer, sheet_name='Full_Statistics', index=False)

    print(f"  -> Saved Individual: {full_path}")

    display_cols = ['Configuration'] + [f'C{i}' for i in range(1, min(7, NUM_CYCLES + 1))]
    print(f"  -> Preview (BV):")
    print(bv_table[display_cols].to_string(index=False))
    print("-" * 40)


if consolidated_bv:
    combined_bv_df = pd.concat(consolidated_bv, ignore_index=True)
    combined_req_df = pd.concat(consolidated_req, ignore_index=True)
    combined_test_df = pd.concat(consolidated_test, ignore_index=True)

    combined_path = os.path.join(RESULTS_DIR, "summary_statistics_ALL_COMBINED.xlsx")

    with pd.ExcelWriter(combined_path, engine='openpyxl') as writer:
        combined_bv_df.to_excel(writer, sheet_name='All_BV_Summaries', index=False)
        combined_req_df.to_excel(writer, sheet_name='All_Req_Summaries', index=False)
        combined_test_df.to_excel(writer, sheet_name='All_Test_Summaries', index=False)

    print("\n" + "="*60)
    print(f"Saved Consolidated File: {combined_path}")

print("\nAll summary statistics calculated and saved successfully!")

In [ ]:
# ======================================================
# Cell 8: Comparative Plots — Hist-RTSGA vs RTSGA vs Random
# ======================================================
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.backends.backend_pdf
import numpy as np
import pandas as pd
import os
import random

RESULTS_DIR    = os.path.join(OUTPUT_DIR, "Comparison_Plots")
STATISTICS_DIR = os.path.join(OUTPUT_DIR, "Statistics")
os.makedirs(RESULTS_DIR, exist_ok=True)

# ------------------------------------------------------
# 1. Load RTSGA Baseline using BASELINE_FILE (set by Cell 1)
# ------------------------------------------------------
rtsga_df   = None
rtsga_path = BASELINE_FILE

if rtsga_path and os.path.exists(rtsga_path):
    try:
        rtsga_df = pd.read_excel(rtsga_path, sheet_name='RTSGA_Full')
        print(f"✅ RTSGA baseline loaded ({rtsga_df.shape[0]} cycles)")
    except Exception as e:
        print(f"⚠️  Could not load RTSGA baseline: {e}")
else:
    print(f"⚠️  RTSGA baseline file not found at: {rtsga_path}")

# ------------------------------------------------------
# 2. Generate Random Selection Baseline
# ------------------------------------------------------
def run_random_instance(max_exec_time, data_maps):
    all_req_ids   = data_maps['all_req_ids']
    req_to_tests  = data_maps['req_to_tests']
    req_to_bv     = data_maps['req_to_bv']
    test_to_time  = data_maps['test_to_time']

    indices       = list(range(len(all_req_ids)))
    random.shuffle(indices)

    covered_tests = set()
    used_time     = 0
    selected_reqs = []

    for idx in indices:
        req       = all_req_ids[idx]
        new_tests = req_to_tests.get(req, set()) - covered_tests
        add_time  = sum(test_to_time.get(t, 0) for t in new_tests)
        if used_time + add_time <= max_exec_time:
            selected_reqs.append(req)
            covered_tests |= new_tests
            used_time     += add_time

    total_bv = sum(req_to_bv.get(r, 0) for r in selected_reqs)
    return selected_reqs, total_bv


def generate_random_statistics(data_maps, rtw_ratio, num_cycles, runs_per_cycle):
    max_exec_time = rtw_ratio * TOTAL_EXEC_TIME
    rows = []

    for cycle_idx in range(1, num_cycles + 1):
        bv_vals, req_vals, test_vals = [], [], []

        for _ in range(runs_per_cycle):
            sel_reqs, bv = run_random_instance(max_exec_time, data_maps)

            bv_vals.append(bv)
            req_vals.append(len(sel_reqs) / data_maps['n_reqs'] * 100)

            covered = set()
            for r in sel_reqs:
                covered |= data_maps['req_to_tests'].get(r, set())
            test_vals.append(len(covered) / data_maps['n_tests'] * 100)

        rows.append({
            'cycle':         cycle_idx,
            'bv_mean':       np.mean(bv_vals),
            'bv_std':        np.std(bv_vals),
            'req_cov_mean':  np.mean(req_vals),
            'req_cov_std':   np.std(req_vals),
            'test_cov_mean': np.mean(test_vals),
            'test_cov_std':  np.std(test_vals),
        })

    return pd.DataFrame(rows)


print("Generating random selection baseline (20 cycles × 30 runs)...")
random_df = generate_random_statistics(
    data_maps      = data_maps,
    rtw_ratio      = 0.05,
    num_cycles     = NUM_CYCLES,
    runs_per_cycle = RUNS_PER_CYCLE,
)
print(f"✅ Random baseline ready ({len(random_df)} cycles)")

# ------------------------------------------------------
# 3. Plotting — 3 bars per cycle
# ------------------------------------------------------
def create_three_way_plot(df_hist, df_rtsga, df_random,
                           config_name, metric_name, metric_key, ylabel):
    n       = len(df_hist)
    x       = np.arange(n)
    width   = 0.25

    fig, ax = plt.subplots(figsize=(16, 7))

    ax.bar(x - width, df_hist[f'{metric_key}_mean'], width,
           label=f'Hist-RTSGA ({config_name})',
           color='#4A90E2', alpha=0.92,
           yerr=df_hist[f'{metric_key}_std'], capsize=3)

    if df_rtsga is not None:
        lim = min(n, len(df_rtsga))
        ax.bar(x[:lim], df_rtsga[f'{metric_key}_mean'].values[:lim], width,
               label='RTSGA (Baseline)',
               color='#FF8C42', alpha=0.92,
               yerr=df_rtsga[f'{metric_key}_std'].values[:lim], capsize=3)

    if df_random is not None:
        lim = min(n, len(df_random))
        ax.bar(x[:lim] + width, df_random[f'{metric_key}_mean'].values[:lim], width,
               label='Random Selection',
               color='#9B9B9B', alpha=0.92,
               yerr=df_random[f'{metric_key}_std'].values[:lim], capsize=3)

    ax.set_xlabel('Regression Cycle (k)', fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=12, fontweight='bold')
    ax.set_title(f'{metric_name}: {config_name} vs RTSGA vs Random',
                 fontsize=14, fontweight='bold', pad=15)
    ax.set_xticks(x)
    ax.set_xticklabels([f'C{int(c)}' for c in df_hist['cycle']], fontsize=9)
    ax.legend(loc='upper right', fontsize=11)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    plt.tight_layout()
    return fig

# ------------------------------------------------------
# 4. Generate & Save
# ------------------------------------------------------
if not all_statistics:
    print("❌ all_statistics is empty — run Cell 6 first.")
else:
    all_figures = []

    for label, df_stats in all_statistics.items():
        print(f"  Plotting: {label}")

        for metric_key, metric_name, ylabel in [
            ('bv',       'Business Value',   'Mean BV'),
            ('req_cov',  'Req Coverage',     'Mean Req Coverage (%)'),
            ('test_cov', 'Test Coverage',    'Mean Test Coverage (%)'),
        ]:
            fig = create_three_way_plot(
                df_hist     = df_stats,
                df_rtsga    = rtsga_df,
                df_random   = random_df,
                config_name = label,
                metric_name = metric_name,
                metric_key  = metric_key,
                ylabel      = ylabel,
            )
            png_name = f'{metric_key}_Compare_{label}.png'
            fig.savefig(os.path.join(RESULTS_DIR, png_name), dpi=300)
            all_figures.append(fig)

    pdf_path = os.path.join(RESULTS_DIR, "All_Comparisons_Combined.pdf")
    try:
        with matplotlib.backends.backend_pdf.PdfPages(pdf_path) as pdf:
            for fig in all_figures:
                pdf.savefig(fig)
        print(f"\n✅ Combined PDF saved → {pdf_path}")
    except Exception as e:
        print(f"\n⚠️  PDF save failed ({e}). Individual PNGs are still saved.")
    finally:
        for fig in all_figures:
            plt.close(fig)

    print("=" * 60)
    print(f"✅ All plots saved → {RESULTS_DIR}")
    print("=" * 60)